# Expert Policy Evaluation
Load and evaluate the trained expert policies (π₁) for all three environments.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from stable_baselines3 import PPO, SAC, TD3
from stable_baselines3.common.evaluation import evaluate_policy

MODELS_DIR = "models"
N_EVAL_EPISODES = 50

## 1. Check which models are available

In [ ]:
MODEL_REGISTRY = {
    "cartpole": {
        "path": os.path.join(MODELS_DIR, "expert_cartpole"),
        "algo": PPO,
        "env_id": "CartPole-v1",
        "label": "CartPole-v1  (PPO)",
        "color": "#2196F3",
        "threshold": 475,
        "optimal": 500,
        "unit": "steps",
    },
    "pendulum": {
        "path": os.path.join(MODELS_DIR, "expert_pendulum"),
        "algo": SAC,
        "env_id": "Pendulum-v1",
        "label": "Pendulum-v1  (SAC)",
        "color": "#E91E63",
        "threshold": -200,
        "optimal": -120,
        "unit": "return",
    },
    "mcc": {
        "path": os.path.join(MODELS_DIR, "expert_mcc"),
        "algo": TD3,
        "env_id": "MountainCarContinuous-v0",
        "label": "MountainCarContinuous-v0  (TD3+OU)",
        "color": "#4CAF50",
        "threshold": 90,
        "optimal": 93,
        "unit": "return",
    },
}

available = {}
for key, cfg in MODEL_REGISTRY.items():
    exists = os.path.exists(cfg["path"] + ".zip")
    status = "✓ found" if exists else "✗ missing"
    print(f"  {cfg['label']:<42}  {status}")
    if exists:
        available[key] = cfg

print(f"\n{len(available)}/{len(MODEL_REGISTRY)} models available.")

## 2. Load models and run evaluation

In [ ]:
results = {}

for key, cfg in available.items():
    print(f"\n{'='*55}")
    print(f"  {cfg['label']}")
    print(f"{'='*55}")

    model = cfg["algo"].load(cfg["path"], device="mps")
    env   = gym.make(cfg["env_id"])

    episode_rewards = []
    episode_lengths = []

    for ep in range(N_EVAL_EPISODES):
        obs, _ = env.reset(seed=ep)
        done = False
        total_r = 0.0
        steps = 0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)
            total_r += reward
            steps += 1
            done = terminated or truncated
        episode_rewards.append(total_r)
        episode_lengths.append(steps)

    env.close()

    mean_r = np.mean(episode_rewards)
    std_r  = np.std(episode_rewards)
    passed = mean_r >= cfg["threshold"]
    status = "PASS" if passed else "FAIL"

    print(f"  Episodes       : {N_EVAL_EPISODES}")
    print(f"  Mean return    : {mean_r:.2f} ± {std_r:.2f}")
    print(f"  Min / Max      : {min(episode_rewards):.2f} / {max(episode_rewards):.2f}")
    print(f"  Mean ep length : {np.mean(episode_lengths):.1f} steps")
    print(f"  Threshold      : {cfg['threshold']}   →  [{status}]")

    results[key] = {
        "rewards": episode_rewards,
        "lengths": episode_lengths,
        "mean": mean_r,
        "std": std_r,
        "passed": passed,
    }

## 3. Return distributions

In [ ]:
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 4))
if n == 1:
    axes = [axes]

for ax, (key, res) in zip(axes, results.items()):
    cfg = available[key]
    ax.hist(res["rewards"], bins=20, color=cfg["color"], edgecolor="white", alpha=0.85)
    ax.axvline(res["mean"],        color="black",  linestyle="--", linewidth=1.5, label=f"Mean: {res['mean']:.1f}")
    ax.axvline(cfg["threshold"],   color="orange", linestyle=":",  linewidth=1.5, label=f"Threshold: {cfg['threshold']}")
    ax.axvline(cfg["optimal"],     color="green",  linestyle=":",  linewidth=1.5, label=f"Optimal: {cfg['optimal']}")
    ax.set_title(cfg["label"], fontsize=10, fontweight="bold")
    ax.set_xlabel("Episode Return")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Expert Policy — Return Distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/expert_return_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Episode-by-episode returns

In [ ]:
fig, axes = plt.subplots(1, n, figsize=(6 * n, 4))
if n == 1:
    axes = [axes]

for ax, (key, res) in zip(axes, results.items()):
    cfg = available[key]
    ax.plot(res["rewards"], color=cfg["color"], linewidth=1.5, alpha=0.8)
    ax.axhline(res["mean"],      color="black",  linestyle="--", linewidth=1.2, label=f"Mean: {res['mean']:.1f}")
    ax.axhline(cfg["threshold"], color="orange", linestyle=":",  linewidth=1.2, label=f"Threshold: {cfg['threshold']}")
    ax.fill_between(range(N_EVAL_EPISODES),
                    res["mean"] - res["std"],
                    res["mean"] + res["std"],
                    color=cfg["color"], alpha=0.15, label="±1 std")
    ax.set_title(cfg["label"], fontsize=10, fontweight="bold")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Return")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Expert Policy — Per-Episode Returns", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/expert_per_episode_returns.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Summary table

In [ ]:
print(f"\n{'Environment':<42} {'Mean':>8} {'Std':>7} {'Min':>8} {'Max':>8} {'Threshold':>10} {'Status':>7}")
print("-" * 95)
for key, res in results.items():
    cfg = available[key]
    status = "PASS" if res["passed"] else "FAIL"
    print(
        f"  {cfg['label']:<40} "
        f"{res['mean']:>8.1f} "
        f"{res['std']:>7.1f} "
        f"{min(res['rewards']):>8.1f} "
        f"{max(res['rewards']):>8.1f} "
        f"{cfg['threshold']:>10} "
        f"{status:>7}"
    )

In [ ]:
import imageio
import base64
from IPython.display import HTML, display

os.makedirs("videos", exist_ok=True)

def record_episode(key, cfg, seed=0):
    model = cfg["algo"].load(cfg["path"], device="mps")
    env   = gym.make(cfg["env_id"], render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    frames, total_r, done = [], 0.0, False
    while not done:
        frames.append(env.render())
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_r += reward
        done = terminated or truncated
    env.close()
    path = f"videos/{key}.mp4"
    imageio.mimwrite(path, frames, fps=30, quality=8)
    return path, total_r

def show_video(path, label, total_r):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f"<h3>{label} — return: {total_r:.1f}</h3>"
        f'<video width="480" controls autoplay loop>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
        f"</video>"
    ))

for key, cfg in available.items():
    print(f"Recording {cfg['label']}...")
    path, total_r = record_episode(key, cfg)
    show_video(path, cfg["label"], total_r)

## 6. Video playback
Records one episode per available model and displays it inline.
Requires `pip install imageio imageio-ffmpeg`.

In [ ]:
import imageio
import base64
from IPython.display import HTML, display

os.makedirs("videos", exist_ok=True)

def record_episode(key, cfg, seed=0):
    model = cfg["algo"].load(cfg["path"], device="mps")
    env   = gym.make(cfg["env_id"], render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    frames, total_r, done = [], 0.0, False
    while not done:
        frames.append(env.render())
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_r += reward
        done = terminated or truncated
    env.close()
    path = f"videos/{key}.mp4"
    imageio.mimwrite(path, frames, fps=30, quality=8)
    return path, total_r

def show_video(path, label, total_r):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f"<h3>{label} — return: {total_r:.1f}</h3>"
        f'<video width="480" controls autoplay loop>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
        f"</video>"
    ))

for key, cfg in available.items():
    print(f"Recording {cfg['label']}...")
    path, total_r = record_episode(key, cfg)
    show_video(path, cfg["label"], total_r)
